# 18 · Comparación con un modelo comercial (DeepSeek)

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Clasifica **la misma prueba final de 10.000 comentarios** con la que se evaluaron los 17 candidatos, ahora con un modelo
comercial (DeepSeek) sin ajuste fino, en tres modos:

| Modo | Qué recibe DeepSeek |
|---|---|
| `zero_shot` | Exactamente la misma plantilla de EmoGol, sin ejemplos (la comparación principal). |
| `guia` | La guía de criterios con que se etiquetaron los datos: definición de las siete emociones, reglas para los casos dudosos y la orden de responder con una sola palabra. |
| `guia_few_shot` | La misma guía más un ejemplo por emoción tomado del conjunto de entrenamiento (nunca de la prueba). |

Los modos `guia` y `guia_few_shot` son una prueba de robustez: le dan a DeepSeek toda la ayuda posible sin ajuste fino.
Cada comentario se envía por separado. Calcula las mismas métricas
(accuracy, F1 macro, IC bootstrap con 1.000 remuestreos y semilla 42, F1 por emoción, inválidas) y la prueba de McNemar
frente a EmoGol-2B, igual que para los demás modelos.

**Antes de empezar**
1. En https://platform.deepseek.com crea una cuenta, recarga saldo (*Top up*) y crea una clave de API (*API keys*). La ejecución completa de los tres modos costó cerca de 1 USD (8 de octubre de 2026).
2. Guarda la clave en los *secretos* de Colab (ícono de llave, a la izquierda) con el nombre `DEEPSEEK_API_KEY` y activa el acceso del notebook. No la pegues en el código.
3. El secreto `HF_TOKEN` (el mismo de los otros notebooks) debe estar activo: la prueba y las predicciones de EmoGol-2B se descargan de Hugging Face.
4. Este notebook **no necesita GPU**: usa el entorno normal de CPU.
5. Ejecuta las celdas en orden con *Entorno de ejecución → Ejecutar todas*.

El avance se guarda en Drive cada 50 comentarios. Si Colab se desconecta o se acaba el saldo, vuelve a ejecutar todo y continúa donde quedó.
Las respuestas `zero_shot` que ya están en Drive se reutilizan: no se vuelven a pedir ni a cobrar.

**Control de gasto:** el notebook calcula el costo de cada consulta con la tarifa real del momento y se detiene solo si los modos nuevos
superan `PRESUPUESTO_USD`. Tampoco arranca en hora pico de DeepSeek, cuando la tarifa se duplica (en Colombia: de domingo a jueves de 8 a 11 p. m.
y de lunes a viernes de 1 a 5 a. m.).

## 1. Dependencias

In [ ]:
!pip install -q -U openai "pandas<3" scikit-learn scipy

## 2. Configuración

In [ ]:
import os
from google.colab import userdata, drive
from huggingface_hub import login, hf_hub_download
import openai
from openai import OpenAI

login(token=userdata.get('HF_TOKEN'))
REPO_DATOS = "santiagortegasegura/emotion-futbol-data"
drive.mount('/content/drive')
CARPETA = "/content/drive/MyDrive/EmoGol_comercial"

client = OpenAI(api_key=userdata.get('DEEPSEEK_API_KEY'), base_url="https://api.deepseek.com",
                timeout=60, max_retries=0)

# Modelos disponibles para tu clave (si la clave está mal, aquí sale el error 401)
for m in client.models.list():
    print(m.id)

In [ ]:
MODELO = "deepseek-flash"     # DeepSeek-V4.1-Flash; debe aparecer en la lista de arriba

EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
MODOS = ['zero_shot', 'guia', 'guia_few_shot']
MODOS_NUEVOS = ['guia', 'guia_few_shot']
PRESUPUESTO_USD = 1.50        # tope de gasto para los modos guia y guia_few_shot
PERMITIR_HORA_PICO = False    # en hora pico la tarifa se duplica; no lo cambies salvo que sepas lo que haces

# DeepSeek no fija un límite de consultas por minuto; este ritmo termina la prueba en unos 20 minutos.
RPM_OBJETIVO = 600
HILOS = 16                    # consultas simultáneas; el ritmo total lo controla RPM_OBJETIVO
GUARDAR_CADA = 50

# Tarifa de deepseek-flash fuera de hora pico, en USD por millón de tokens (https://api-docs.deepseek.com/quick_start/pricing).
# En hora pico (01-04 y 06-10 UTC, de lunes a viernes) todo vale el doble.
PRECIO_ENTRADA_USD_M = 0.15         # entrada sin caché
PRECIO_ENTRADA_CACHE_USD_M = 0.003  # entrada repetida que DeepSeek ya tenía en caché (la guía se repite en cada consulta)
PRECIO_SALIDA_USD_M = 0.60          # incluye los tokens de razonamiento, si los hubiera

from datetime import datetime, timezone
def hora_pico(t=None):
    t = t or datetime.now(timezone.utc)
    return t.weekday() < 5 and (1 <= t.hour < 4 or 6 <= t.hour < 10)

if hora_pico() and not PERMITIR_HORA_PICO:
    raise RuntimeError("Ahora es hora pico de DeepSeek y la tarifa se duplica. Vuelve a ejecutar más tarde "
                       "(en Colombia, evita de domingo a jueves de 8 a 11 p. m. y de lunes a viernes de 1 a 5 a. m.).")
os.makedirs(CARPETA, exist_ok=True)
print("Modelo:", MODELO)

## 3. Prueba final y predicciones de EmoGol-2B

In [ ]:
import pandas as pd, numpy as np, time, json

def bajar(nombre):
    return pd.read_csv(hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename=nombre), keep_default_na=False)

prueba = bajar("final/prueba_final_10k.csv")
pred_emogol = bajar("final/predicciones/07_predicciones_gemma2_2b.csv")
assert len(prueba) == len(pred_emogol) == 10000
assert (prueba.text.astype(str).values == pred_emogol.text.astype(str).values).all(), "El orden no coincide."
muestra = pd.DataFrame({'id_prueba': np.arange(len(prueba)), 'text': prueba.text.astype(str),
                        'label': prueba.label, 'pred_emogol': pred_emogol.prediccion})
print(f"EmoGol-2B en la prueba: accuracy {np.mean(muestra.label == muestra.pred_emogol):.4f}")   # debe dar 0.8248

## 4. Prompts y regla de lectura (las mismas de EmoGol)

In [ ]:
import unicodedata

INSTRUCCION = ("Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: "
               f"{', '.join(EMOCIONES)}.")

def prompt_zero_shot(texto):                     # idéntico a formatear_prompt_inferencia del notebook 10
    return f"{INSTRUCCION}\n\nComentario: {texto}\n\nEmoción:"

# Guía de criterios: las definiciones y las reglas de desambiguación con que se etiquetó el conjunto de datos
GUIA = (f"{INSTRUCCION}\n\n"
        "Definiciones:\n"
        "- alegria: felicidad, celebración, orgullo o elogio genuino a un jugador, un equipo o una jugada.\n"
        "- tristeza: dolor o decepción del propio autor, con una señal explícita de pena.\n"
        "- enojo: rabia, indignación, insultos o quejas contra jugadores, árbitros, técnicos, dirigentes o el propio equipo.\n"
        "- miedo: nervios, angustia o temor que expresa el propio autor por lo que puede pasar.\n"
        "- sorpresa: asombro o incredulidad ante algo inesperado, con una señal explícita.\n"
        "- burla: mofa, sarcasmo o ironía para ridiculizar a un rival, un jugador, un árbitro u otra afición.\n"
        "- neutral: comentario informativo, analítico, pregunta u opinión sin una emoción clara.\n\n"
        "Reglas para los casos dudosos:\n"
        "1. Risa escrita (jaja, jsjs, xd) o emojis de risa junto a una crítica: burla.\n"
        "2. Elogio irónico, pregunta retórica de desprecio o exageración que ridiculiza: burla.\n"
        "3. Elogio genuino: alegria, no sorpresa.\n"
        "4. Usa sorpresa solo si hay una señal explícita de asombro.\n"
        "5. Comentario informativo o analítico: neutral.\n"
        "6. Usa miedo solo si el propio autor expresa nervios o temor.\n"
        "7. Usa tristeza solo si hay una señal explícita de dolor propio; una queja sin esa señal es enojo.\n"
        "8. Emoji de payaso: burla.\n\n"
        "Responde únicamente con la categoría, en una sola palabra, en minúsculas y sin tildes.")

def prompt_guia(texto):
    return f"{GUIA}\n\nComentario: {texto}\n\nEmoción:"

# Un ejemplo por emoción, tomado del conjunto de entrenamiento (semilla 42); nunca de la prueba
from huggingface_hub import list_repo_files
archivos = [a for a in list_repo_files(REPO_DATOS, repo_type="dataset") if a.endswith('.csv')]
candidatos = sorted([a for a in archivos if any(k in a.lower() for k in ('entren', 'train'))], key=lambda a: (not a.startswith('final/'), len(a)))
assert candidatos, f"No encontré el archivo de entrenamiento. Archivos del repositorio: {archivos}"
ARCHIVO_ENTRENAMIENTO = candidatos[0]
ent = bajar(ARCHIVO_ENTRENAMIENTO)
assert {'text', 'label'} <= set(ent.columns), f"Columnas inesperadas en {ARCHIVO_ENTRENAMIENTO}: {list(ent.columns)}"
ent = ent[~ent.text.astype(str).isin(set(muestra.text))]
largo = ent.text.astype(str).str.len()
ent = ent[(largo >= 25) & (largo <= 110) & ~ent.text.astype(str).str.contains('http')]
if 'confianza' in ent.columns:
    ent = ent[pd.to_numeric(ent.confianza, errors='coerce') >= pd.to_numeric(ent.confianza, errors='coerce').max()]
EJEMPLOS = pd.concat([ent[ent.label == e].sample(1, random_state=42) for e in EMOCIONES]).sample(frac=1, random_state=42)
BLOQUE_EJEMPLOS = "Ejemplos:\n\n" + "\n\n".join(f"Comentario: {t}\nEmoción: {l}" for t, l in zip(EJEMPLOS.text, EJEMPLOS.label))

def prompt_guia_few_shot(texto):
    return f"{GUIA}\n\n{BLOQUE_EJEMPLOS}\n\nComentario: {texto}\n\nEmoción:"

PROMPTS = {'zero_shot': prompt_zero_shot, 'guia': prompt_guia, 'guia_few_shot': prompt_guia_few_shot}
print("Ejemplos tomados de:", ARCHIVO_ENTRENAMIENTO, "\n")

def sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def parsear_estricta(generado):                  # primera palabra; se ignoran mayúsculas y tildes
    limpio = generado.strip().lower().split()[0] if generado.strip() else ""
    limpio = sin_tildes(limpio.strip(".,;:!¡¿?\"'*"))
    return limpio if limpio in EMOCIONES else "invalido"

def parsear_flexible(generado):                  # primera emoción en cualquier parte de la respuesta
    t = sin_tildes(generado.lower())
    pos = {e: t.find(e) for e in EMOCIONES if e in t}
    return min(pos, key=pos.get) if pos else "invalido"

print(prompt_guia_few_shot("Qué golazo, no lo puedo creer"))

## 5. Llamada al modelo

In [ ]:
import threading

def costo(entrada, cache, salida):              # tarifa fuera de hora pico
    return ((entrada - cache) * PRECIO_ENTRADA_USD_M + cache * PRECIO_ENTRADA_CACHE_USD_M + salida * PRECIO_SALIDA_USD_M) / 1e6

def pedir(prompt, extra):
    return client.chat.completions.create(model=MODELO, messages=[{"role": "user", "content": prompt}],
                                          temperature=0, **extra)

# Se busca la configuración más parecida a EmoGol: temperatura 0 y sin razonamiento previo.
CANDIDATAS = [
    ("sin razonamiento (thinking disabled)", dict(max_tokens=16, extra_body={"thinking": {"type": "disabled"}})),
    ("sin razonamiento (reasoning_effort=none)", dict(max_tokens=16, reasoning_effort="none")),
    ("configuración por defecto", dict(max_tokens=2048)),
]
CONFIG, NOMBRE_CONFIG = None, None
for nombre, extra in CANDIDATAS:
    try:
        r = pedir(prompt_zero_shot("Qué golazo"), extra)
        CONFIG, NOMBRE_CONFIG = extra, nombre
        print("Respuesta de prueba:", repr(r.choices[0].message.content))
        break
    except openai.AuthenticationError as e:
        raise RuntimeError("La clave DEEPSEEK_API_KEY no es válida. Revísala en los secretos de Colab.") from e
    except openai.APIStatusError as e:
        if e.status_code == 402:
            raise RuntimeError("Tu cuenta de DeepSeek no tiene saldo. Recarga en platform.deepseek.com (Top up).") from e
        if e.status_code in (429, 500, 503):
            raise RuntimeError(f"DeepSeek está ocupado (error {e.status_code}). Espera unos minutos y vuelve a ejecutar.") from e
        print(f"No acepta '{nombre}': {str(e)[:120]}")
assert CONFIG is not None, "El modelo no aceptó ninguna configuración."
print("Configuración usada:", NOMBRE_CONFIG)

class DetenerEjecucion(Exception):
    pass

_turno = {'siguiente': 0.0}
_candado_turno = threading.Lock()

def esperar_turno():                       # reparte las consultas para no pasar de RPM_OBJETIVO
    with _candado_turno:
        ahora = time.time()
        inicio = max(ahora, _turno['siguiente'])
        _turno['siguiente'] = inicio + 60.0 / RPM_OBJETIVO
    time.sleep(max(0.0, inicio - ahora))

def clasificar(prompt, intentos=8):
    espera = 5
    for _ in range(intentos):
        try:
            esperar_turno()
            t0 = time.perf_counter()
            r = pedir(prompt, CONFIG)
            lat = time.perf_counter() - t0
            u = r.usage
            det = getattr(u, 'completion_tokens_details', None)
            razon = (getattr(det, 'reasoning_tokens', 0) or 0) if det else 0
            cache = getattr(u, 'prompt_cache_hit_tokens', 0) or 0
            pico = hora_pico()
            usd = costo(u.prompt_tokens, cache, u.completion_tokens) * (2 if pico else 1)
            return dict(respuesta=r.choices[0].message.content or "", fin=str(r.choices[0].finish_reason), latencia_s=lat,
                        tokens_entrada=u.prompt_tokens, tokens_entrada_cache=cache,
                        tokens_salida=u.completion_tokens - razon, tokens_razonamiento=razon, hora_pico=pico, usd=usd)
        except openai.AuthenticationError as e:
            raise DetenerEjecucion("La clave dejó de ser válida (error 401).") from e
        except openai.APIStatusError as e:
            if e.status_code == 402:
                raise DetenerEjecucion("Se acabó el saldo de DeepSeek (error 402). Recarga y vuelve a ejecutar.")
            if e.status_code not in (429, 500, 503):     # rechazo del propio modelo: cuenta como respuesta inválida
                return dict(respuesta="", fin=f"ERROR_{e.status_code}", latencia_s=np.nan, tokens_entrada=0,
                            tokens_entrada_cache=0, tokens_salida=0, tokens_razonamiento=0, hora_pico=hora_pico(), usd=0.0)
        except (openai.APIConnectionError, openai.APITimeoutError):
            pass
        time.sleep(espera); espera = min(espera * 2, 120)
    raise DetenerEjecucion("Demasiados rechazos seguidos; DeepSeek puede estar saturado. Espera y vuelve a ejecutar.")

## 6. Clasificación (se puede interrumpir y reanudar)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def ruta(modo):
    return f"{CARPETA}/18_respuestas_{MODELO}_{modo}.csv"

def leer(modo):
    return pd.read_csv(ruta(modo), keep_default_na=False) if os.path.exists(ruta(modo)) else pd.DataFrame()

def gasto(df):                                   # USD gastados en un archivo de respuestas
    if not len(df):
        return 0.0
    if 'usd' in df.columns:
        return float(pd.to_numeric(df.usd, errors='coerce').fillna(0).sum())
    return costo(df.tokens_entrada.sum(), df.tokens_entrada_cache.sum(), df.tokens_salida.sum() + df.tokens_razonamiento.sum())

parar = threading.Event()
motivo = {'texto': ''}

def correr(modo):
    hechas = leer(modo)
    gasto_otros = sum(gasto(leer(m)) for m in MODOS_NUEVOS if m != modo)
    listos = set(hechas.id_prueba) if len(hechas) else set()
    pendientes = muestra[~muestra.id_prueba.isin(listos)]
    print(f"{modo}: {len(listos)} hechas, {len(pendientes)} pendientes de {len(muestra)}")
    filas, candado = [], threading.Lock()

    def guardar():
        nonlocal hechas, filas
        if filas:
            hechas = pd.concat([hechas, pd.DataFrame(filas)], ignore_index=True)
            hechas.to_csv(ruta(modo), index=False); filas = []

    acumulado = {'usd': gasto(hechas) + gasto_otros}
    if modo in MODOS_NUEVOS and acumulado['usd'] >= PRESUPUESTO_USD:
        print(f"{modo}: no se ejecuta, ya se alcanzó el presupuesto ({acumulado['usd']:.2f} USD).")
        return len(hechas) == len(muestra)

    def uno(fila):
        if parar.is_set():
            return
        try:
            res = clasificar(PROMPTS[modo](fila.text))
        except DetenerEjecucion as e:
            motivo['texto'] = str(e); parar.set(); return
        with candado:
            filas.append(dict(id_prueba=fila.id_prueba, **res))
            acumulado['usd'] += res.get('usd', 0.0)
            if modo in MODOS_NUEVOS and acumulado['usd'] >= PRESUPUESTO_USD:
                motivo['texto'] = f"Se alcanzó el presupuesto de {PRESUPUESTO_USD:.2f} USD."; parar.set()
            if len(filas) >= GUARDAR_CADA:
                guardar()
                if len(hechas) % 500 == 0:
                    print(f"  {modo}: {len(hechas)}/{len(muestra)}  gasto acumulado: {acumulado['usd']:.2f} USD", flush=True)

    parar.clear()
    try:
        with ThreadPoolExecutor(HILOS) as ex:
            list(ex.map(uno, pendientes.itertuples()))
    finally:
        with candado:
            guardar()
    if parar.is_set():
        print("\nSe detuvo la clasificación. El avance quedó guardado.", motivo['texto'])
    print(f"\n{modo}: {len(hechas)}/{len(muestra)} guardadas en Drive")
    return len(hechas) == len(muestra)

completos = {}
for m in MODOS:
    completos[m] = correr(m)
    if parar.is_set():
        break
print(completos)
print(f"Gasto total de los modos nuevos: {sum(gasto(leer(m)) for m in MODOS_NUEVOS):.2f} USD")

## 7. Métricas y comparación con EmoGol-2B (misma prueba, mismas métricas)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from scipy.stats import binomtest, chi2

assert len(completos) == len(MODOS) and all(completos.values()), "Aún faltan respuestas: vuelve a ejecutar la sección 6."

def f1_macro_np(yt, yp):
    k = len(EMOCIONES); idx = {e: i for i, e in enumerate(EMOCIONES)}
    a = np.array([idx[x] for x in yt]); b = np.array([idx.get(x, k) for x in yp])
    m = np.bincount(a * (k + 1) + b, minlength=k * (k + 1)).reshape(k, k + 1)
    tp = np.diag(m[:, :k]); p = m[:, :k].sum(0); r = m.sum(1)
    f = np.where(p + r > 0, 2 * tp / np.maximum(p + r, 1), 0)
    return f.mean()

def ic(valores):
    return [float(np.percentile(valores, 2.5)), float(np.percentile(valores, 97.5))]

y = muestra.label.to_numpy()
pred = {'emogol_2b': muestra.pred_emogol.to_numpy()}
extra = {}
for modo in MODOS:
    r = pd.read_csv(ruta(modo), keep_default_na=False).drop_duplicates('id_prueba').set_index('id_prueba').loc[muestra.id_prueba]
    pred[modo] = r.respuesta.astype(str).map(parsear_estricta).to_numpy()
    extra[modo] = dict(
        flexible=r.respuesta.astype(str).map(parsear_flexible).to_numpy(),
        no_completadas=int((~r.fin.str.lower().str.contains('stop')).sum()),
        latencia_prom_s=float(pd.to_numeric(r.latencia_s, errors='coerce').mean()),
        tokens_entrada=int(r.tokens_entrada.sum()), tokens_entrada_cache=int(r.tokens_entrada_cache.sum()),
        tokens_salida=int(r.tokens_salida.sum()),
        tokens_razonamiento=int(r.tokens_razonamiento.sum()),
        usd=gasto(r.reset_index()),
        ejemplos_respuesta=r.respuesta.head(5).tolist())

rng = np.random.default_rng(42); B = 1000
muestras_boot = [rng.integers(0, len(y), len(y)) for _ in range(B)]
filas = []
for nombre, yp in pred.items():
    p, rc, f, _ = precision_recall_fscore_support(y, yp, labels=EMOCIONES, zero_division=0)
    fila = dict(modelo=nombre, n=len(y), accuracy=accuracy_score(y, yp),
                accuracy_ic95=ic([np.mean(y[i] == yp[i]) for i in muestras_boot]),
                f1_macro=f1_macro_np(y, yp), f1_macro_ic95=ic([f1_macro_np(y[i], yp[i]) for i in muestras_boot]),
                precision_macro=p.mean(), recall_macro=rc.mean(),
                invalidas=int((yp == 'invalido').sum()), errores=int((yp != y).sum()),
                **{f"f1_{e}": float(v) for e, v in zip(EMOCIONES, f)})
    if nombre in extra:
        ex = extra[nombre]
        fila.update(accuracy_flexible=accuracy_score(y, ex['flexible']), f1_macro_flexible=f1_macro_np(y, ex['flexible']),
                    **{k: v for k, v in ex.items() if k not in ('flexible', 'ejemplos_respuesta', 'usd')})
        # EmoGol frente a este modo: McNemar exacto y razones con IC bootstrap pareado
        a_ok, b_ok = pred['emogol_2b'] == y, yp == y
        b_, c_ = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())   # b: solo EmoGol acierta; c: solo DeepSeek
        x2 = (abs(b_ - c_) - 1) ** 2 / (b_ + c_) if b_ + c_ else 0.0          # misma fórmula de la Tabla de McNemar
        fila.update(mcnemar_b=b_, mcnemar_c=c_, mcnemar_chi2=x2, mcnemar_p=float(chi2.sf(x2, 1)),
                    mcnemar_p_exacta=binomtest(b_, b_ + c_, 0.5).pvalue if b_ + c_ else 1.0,
                    razon_errores_comercial_sobre_emogol=fila['errores'] / max(int((~a_ok).sum()), 1),
                    razon_f1_emogol_sobre_comercial=f1_macro_np(y, pred['emogol_2b']) / max(fila['f1_macro'], 1e-9),
                    razon_f1_ic95=ic([f1_macro_np(y[i], pred['emogol_2b'][i]) / max(f1_macro_np(y[i], yp[i]), 1e-9) for i in muestras_boot]))
        usd = ex['usd']
        fila.update(usd_total=usd, usd_por_1000=1000 * usd / len(y))
    filas.append(fila)

res = pd.DataFrame(filas)
res.insert(1, 'modelo_comercial', MODELO); res.insert(2, 'configuracion', NOMBRE_CONFIG)
res.to_csv(f"{CARPETA}/18_resultados_comercial.csv", index=False)

pd.DataFrame({'id_prueba': muestra.id_prueba, 'text': muestra.text, 'label': y, **{f"pred_{k}": v for k, v in pred.items()}}) \
  .to_csv(f"{CARPETA}/18_predicciones_comercial.csv", index=False)

resumen = dict(modelo=MODELO, configuracion=NOMBRE_CONFIG, fecha=time.strftime('%Y-%m-%d'),
               guia=GUIA, archivo_ejemplos=ARCHIVO_ENTRENAMIENTO,
               ejemplos_few_shot=[dict(text=t, label=l) for t, l in zip(EJEMPLOS.text, EJEMPLOS.label)],
               resultados=json.loads(res.to_json(orient='records')),
               ejemplos_respuesta={m: extra[m]['ejemplos_respuesta'] for m in MODOS})
json.dump(resumen, open(f"{CARPETA}/18_resumen_comercial.json", 'w'), ensure_ascii=False, indent=2)

pd.set_option('display.width', 200)
print(res[['modelo', 'accuracy', 'f1_macro', 'invalidas', 'errores']].round(3).to_string(index=False))
print("\nListo. Envía al chat los archivos 18_resultados_comercial.csv, 18_predicciones_comercial.csv y 18_resumen_comercial.json de tu carpeta EmoGol_comercial.")

## Resultados obtenidos (8 de octubre de 2026)

| Modelo y modo | Accuracy | F1 macro [IC 95 %] | Inválidas | Errores frente a EmoGol-2B |
|---|---|---|---|---|
| **EmoGol-2B** (ajuste fino) | **0,825** | **0,776** [0,748; 0,799] | 169 | 1,00 |
| DeepSeek-V4.1-Flash, zero-shot | 0,558 | 0,518 [0,490; 0,548] | 1.638 | 2,53 |
| DeepSeek-V4.1-Flash, guía | 0,649 | 0,555 [0,530; 0,577] | 0 | 2,01 |
| DeepSeek-V4.1-Flash, guía y few-shot | 0,660 | 0,558 [0,535; 0,580] | 0 | 1,94 |

Análisis completo en la Sección 3.10 de la monografía.